# Dynamical and hybrid conditioning for compositional image retrieval

1. Download the necessary libraries
2. Setup the CelebA dataset
3. Download the CLIP model off of HuggingFace
4. Compute and save to disk the dataset embeddings
5. Define the optimized SLERP retrieval
6. Define the retrieval metrics
7. Evaluate performance


## Tests

1. SLERP (a=0.8)
2. modified SLERP (a=0.8)
3. modified SLERP (a=0.6)
4. not even SLERP (a=0.6)

## Download the necessary libraries

In [1]:
%pip install -q torch torchvision tensorboard ftfy regex tqdm scikit-learn scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


In [2]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
import skimage
import torch
import torch.nn as nn
import torchvision
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

## Setup the CelebA dataset

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
!mkdir /content/datasets

In [5]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

In [6]:
import torch
from pathlib import Path

from torchvision.datasets import CelebA

In [7]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")

In [8]:
celeba = CelebA(root=data_root, split="test", download=False)

In [9]:
# This should be 19,962
len(celeba)

19962

## Download the CLIP model off of HuggingFace

In [11]:
from transformers import AutoProcessor, AutoModel

# Download the CLIP ViT-B/32 model
model_name = "openai/clip-vit-base-patch32"

# Load the processor (tokenizer and image preprocessor)
processor = AutoProcessor.from_pretrained(model_name)

# Load the model
model = AutoModel.from_pretrained(model_name)

# Set the model to eval mode
model = model.cuda().eval()

print(f"Successfully downloaded and loaded {model_name} model and processor.")

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Successfully downloaded and loaded openai/clip-vit-base-patch32 model and processor.


## Compute and save to disk the dataset embeddings

In [12]:
def encode_celeba_images(celeba_dataset, processor, model, device="cuda"):

    all_image_embeddings = []
    model.to(device) # Ensure model is on the correct device
    model.eval()     # Set model to evaluation mode

    # Use tqdm for a progress bar to show progress during encoding
    for i in tqdm(range(len(celeba_dataset)), desc="Encoding CelebA images"):
        image, _ = celeba_dataset[i] # CelebA returns (image, target)

        # Preprocess the image using the processor from transformers
        # The processor handles transformations like resizing, normalization, etc.
        inputs = processor(images=image, return_tensors="pt").to(device)

        # Encode the image using the model from transformers
        with torch.no_grad():
            image_features = model.get_image_features(pixel_values=inputs.pixel_values)

        # FIX: Access .pooler_output as image_features is BaseModelOutputWithPooling
        all_image_embeddings.append(image_features.pooler_output.cpu()) # Move to CPU to prevent GPU memory issues during accumulation

    return torch.cat(all_image_embeddings, dim=0)


In [14]:
import torch
import os
from tqdm import tqdm

# Define paths for saving
local_save_path = "/content/celeba_image_embeddings.pt"
drive_save_path = "/content/drive/MyDrive/datasets/celeba_image_embeddings.pt"

# Check if embeddings already exist on Google Drive
if os.path.exists(drive_save_path):
    print(f"Loading image embeddings from Google Drive: {drive_save_path}")
    celeba_image_embeddings = torch.load(drive_save_path)
    print("Image embeddings loaded successfully.")
elif os.path.exists(local_save_path):
    print(f"Loading image embeddings from local path: {local_save_path}")
    celeba_image_embeddings = torch.load(local_save_path)
    print("Image embeddings loaded successfully.")
else:
    print("Image embeddings not found. Computing and saving them...")
    # Call the function to encode CelebA images (it is defined in a previous cell)
    celeba_image_embeddings = encode_celeba_images(celeba_dataset=celeba, processor=processor, model=model, device="cuda")

    # Save locally
    torch.save(celeba_image_embeddings, local_save_path)
    print(f"Image embeddings saved locally to: {local_save_path}")

    # Ensure the directory exists in Google Drive before saving
    drive_dir = os.path.dirname(drive_save_path)
    if not os.path.exists(drive_dir):
        os.makedirs(drive_dir)
        print(f"Created directory on Google Drive: {drive_dir}")

    # Save to Google Drive
    torch.save(celeba_image_embeddings, drive_save_path)
    print(f"Image embeddings saved to Google Drive: {drive_save_path}")


Loading image embeddings from Google Drive: /content/drive/MyDrive/datasets/celeba_image_embeddings.pt
Image embeddings loaded successfully.


In [15]:
import torch

def get_text_embedding(text, processor, model, device="cuda"):
    """
    Generates CLIP text embedding for the given text.
    (Fixed for BaseModelOutputWithPooling output and now returns [512] shape)
    """
    model.to(device)
    model.eval()
    inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    # FIX: Access .pooler_output and then squeeze to remove the batch dimension
    return text_features.pooler_output.squeeze(0).cpu()

## Define the SLERP-based retrieval function

In [16]:
import json

annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")

with open(annotations_path, "r") as f:
    annotations = json.load(f)

len(annotations)

14

In [30]:
# SLERP parameter
slerp_alpha = 0.8

In [31]:
def slerp(v, w, alpha, eps=1e-8):
    """
    Spherical linear interpolation between v and w.

    v, w: tensors of shape [D]
    alpha: interpolation factor in [0, 1]
    """
    # normalize vectors (important for CLIP embeddings)
    v_norm = v / (v.norm(p=2) + eps)
    w_norm = w / (w.norm(p=2) + eps)

    # cosine of angle
    cos_theta = torch.clamp(torch.dot(v_norm, w_norm), -1.0, 1.0)

    theta = torch.acos(cos_theta)

    # if vectors are very close → fallback to linear interpolation
    if theta.abs() < 1e-6:
        return (1 - alpha) * v + alpha * w

    sin_theta = torch.sin(theta)

    a = torch.sin((1 - alpha) * theta) / sin_theta
    b = torch.sin(alpha * theta) / sin_theta

    return a * v + b * w

In [32]:
import torch
import torch.nn.functional as F

def slerp_retrieval(query_index, query_source_image_index, celeba_image_embeddings, K, alpha):

  if type(query_source_image_index) is int:
    query_source_image_index = str(query_source_image_index)

  # Check that the source image index belongs to the example in that query
  # so that we will then be able to check its corresponding ground truth
  try:
    value = annotations[query_index]['ground_truth'][query_source_image_index]
  except (IndexError, KeyError, TypeError) as e:
    return {"error": str(e)}

  # get the original image embedding
  # This will have shape [512]
  query_source_image_embedding = celeba_image_embeddings[int(query_source_image_index)]

  combined_text_embedding = torch.zeros(512)
  for attribute in annotations[query_index]['query'].split(','):

    # extract the correct attribute
    attribute = attribute.strip()
    is_it_positive = attribute[0] =='+'
    attribute = attribute[1:]

    # attribute_embedding will now have shape [512] due to .squeeze(0) in get_text_embedding
    attribute_embedding = get_text_embedding(attribute, processor, model)
    attribute_embedding = F.normalize(attribute_embedding, dim=0)

    if is_it_positive:
        combined_text_embedding += attribute_embedding
    else:
        combined_text_embedding -= attribute_embedding
  combined_text_embedding = F.normalize(combined_text_embedding, dim=0)

  # normalize vectors before slerp
  v = F.normalize(query_source_image_embedding, dim=0)
  w = F.normalize(combined_text_embedding, dim=0)

  # SLERP interpolation
  computed_embedding = slerp(v, w, alpha)

  # Normalize the computed embedding
  normalized_computed_embedding = F.normalize(computed_embedding.unsqueeze(0), p=2, dim=1)

  # Normalize all celeba image embeddings
  normalized_celeba_embeddings = F.normalize(celeba_image_embeddings, p=2, dim=1)

  # Compute cosine similarity between the computed_embedding and all celeba_image_embeddings
  # The result will be a tensor of shape [1, num_images]
  similarities = torch.mm(normalized_computed_embedding, normalized_celeba_embeddings.T)

  # Get the top K most similar images (indices)
  # .squeeze() converts [1, num_images] to [num_images]
  # .topk(K) returns a tuple (values, indices)
  top_k_similarities, top_k_indices = torch.topk(similarities.squeeze(), K)

  # Return the indices of the K most similar images
  return top_k_indices.tolist()

In [40]:
import torch
import torch.nn.functional as F

def modified_slerp_combination(query_index, query_source_image_index, celeba_image_embeddings, K, alpha):

  if type(query_source_image_index) is int:
    query_source_image_index = str(query_source_image_index)

  # Check that the source image index belongs to the example in that query
  try:
    value = annotations[query_index]['ground_truth'][query_source_image_index]
  except (IndexError, KeyError, TypeError) as e:
    return {"error": str(e)}

  # get the original image embedding
  query_source_image_embedding = celeba_image_embeddings[int(query_source_image_index)]

  combined_text_embedding = torch.zeros(512)

  for attribute in annotations[query_index]['query'].split(','):

    attribute = attribute.strip()

    is_it_positive = attribute[0] == '+'
    attribute = attribute[1:]

    if is_it_positive:
      # +attribute → normal embedding
      attribute_embedding = get_text_embedding(attribute, processor, model)

    else:
      # -attribute → use "not attribute"
      negated_text = f"not {attribute}"
      attribute_embedding = get_text_embedding(negated_text, processor, model)

    attribute_embedding = F.normalize(attribute_embedding, dim=0)
    combined_text_embedding += attribute_embedding

  combined_text_embedding = F.normalize(combined_text_embedding, dim=0)

  # normalize vectors before slerp
  v = F.normalize(query_source_image_embedding, dim=0)
  w = F.normalize(combined_text_embedding, dim=0)

  # SLERP interpolation
  computed_embedding = slerp(v, w, alpha)

  # Normalize the computed embedding
  normalized_computed_embedding = F.normalize(
      computed_embedding.unsqueeze(0), p=2, dim=1
  )

  # Normalize all celeba image embeddings
  normalized_celeba_embeddings = F.normalize(celeba_image_embeddings, p=2, dim=1)

  # cosine similarity
  similarities = torch.mm(normalized_computed_embedding, normalized_celeba_embeddings.T)

  # top-k retrieval
  top_k_similarities, top_k_indices = torch.topk(similarities.squeeze(), K)

  return top_k_indices.tolist()

## Define the retrieval metrics

In [33]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
# predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
# ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
# print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
# print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

## Evaluate performance

In [34]:
def metrics_per_query_slerp_retrieval(query_index, celeba_image_embeddings, K):
  total_recall = 0.0
  total_precision = 0.0
  evaluation_count = 0

  for test_image_index, _ in annotations[query_index]['ground_truth'].items():
      predictions = slerp_retrieval(query_index, test_image_index, celeba_image_embeddings, K, slerp_alpha)

      # Handle potential errors from simple_retrieval
      if isinstance(predictions, dict) and "error" in predictions:
            print(f"Warning: simple_retrieval returned an error for query_index {query_index}, source image '{test_image_index}': {predictions['error']}")
            continue # Skip this evaluation if an error occurred

      metrics = evaluate_retrieval(predictions, annotations[query_index]['ground_truth'][str(test_image_index)], K)

      total_recall += metrics[f"Recall@{K}"]
      total_precision += metrics[f"Precision@{K}"]
      evaluation_count += 1

  if evaluation_count > 0:
      average_recall = total_recall / evaluation_count
      average_precision = total_precision / evaluation_count
      print(f"Query Index {query_index} (K={K}):")
      print(f"  Average Recall@{K}: {average_recall:.4f}")
      print(f"  Average Precision@{K}: {average_precision:.4f}")
  else:
      print(f"Query Index {query_index} (K={K}): No evaluations performed.")

In [35]:
for query_index in range(len(annotations)):
  metrics_per_query_slerp_retrieval(query_index, celeba_image_embeddings, 10)

Query Index 0 (K=10):
  Average Recall@10: 0.2079
  Average Precision@10: 0.0303
Query Index 1 (K=10):
  Average Recall@10: 0.3051
  Average Precision@10: 0.0412
Query Index 2 (K=10):
  Average Recall@10: 0.1170
  Average Precision@10: 0.0142
Query Index 3 (K=10):
  Average Recall@10: 0.0571
  Average Precision@10: 0.0071


KeyboardInterrupt: 

In [41]:
def metrics_per_query_modified_slerp_retrieval(query_index, celeba_image_embeddings, K):
  total_recall = 0.0
  total_precision = 0.0
  evaluation_count = 0

  for test_image_index, _ in annotations[query_index]['ground_truth'].items():
      predictions = modified_slerp_combination(query_index, test_image_index, celeba_image_embeddings, K, slerp_alpha)

      # Handle potential errors from simple_retrieval
      if isinstance(predictions, dict) and "error" in predictions:
            print(f"Warning: simple_retrieval returned an error for query_index {query_index}, source image '{test_image_index}': {predictions['error']}")
            continue # Skip this evaluation if an error occurred

      metrics = evaluate_retrieval(predictions, annotations[query_index]['ground_truth'][str(test_image_index)], K)

      total_recall += metrics[f"Recall@{K}"]
      total_precision += metrics[f"Precision@{K}"]
      evaluation_count += 1

  if evaluation_count > 0:
      average_recall = total_recall / evaluation_count
      average_precision = total_precision / evaluation_count
      print(f"Query Index {query_index} (K={K}):")
      print(f"  Average Recall@{K}: {average_recall:.4f}")
      print(f"  Average Precision@{K}: {average_precision:.4f}")
  else:
      print(f"Query Index {query_index} (K={K}): No evaluations performed.")

In [42]:
for query_index in range(len(annotations)):
  metrics_per_query_modified_slerp_retrieval(query_index, celeba_image_embeddings, 10)

Query Index 0 (K=10):
  Average Recall@10: 0.2079
  Average Precision@10: 0.0303
Query Index 1 (K=10):
  Average Recall@10: 0.3051
  Average Precision@10: 0.0412
Query Index 2 (K=10):
  Average Recall@10: 0.0839
  Average Precision@10: 0.0093
Query Index 3 (K=10):
  Average Recall@10: 0.0571
  Average Precision@10: 0.0071


KeyboardInterrupt: 

In [44]:
def metrics_per_query_slerp_retrieval_alpha_06(query_index, celeba_image_embeddings, K):
  total_recall = 0.0
  total_precision = 0.0
  evaluation_count = 0

  for test_image_index, _ in annotations[query_index]['ground_truth'].items():
      predictions = slerp_retrieval(query_index, test_image_index, celeba_image_embeddings, K, 0.6)

      # Handle potential errors from simple_retrieval
      if isinstance(predictions, dict) and "error" in predictions:
            print(f"Warning: simple_retrieval returned an error for query_index {query_index}, source image '{test_image_index}': {predictions['error']}")
            continue # Skip this evaluation if an error occurred

      metrics = evaluate_retrieval(predictions, annotations[query_index]['ground_truth'][str(test_image_index)], K)

      total_recall += metrics[f"Recall@{K}"]
      total_precision += metrics[f"Precision@{K}"]
      evaluation_count += 1

  if evaluation_count > 0:
      average_recall = total_recall / evaluation_count
      average_precision = total_precision / evaluation_count
      print(f"Query Index {query_index} (K={K}):")
      print(f"  Average Recall@{K}: {average_recall:.4f}")
      print(f"  Average Precision@{K}: {average_precision:.4f}")
  else:
      print(f"Query Index {query_index} (K={K}): No evaluations performed.")

In [45]:
for query_index in range(len(annotations)):
  metrics_per_query_slerp_retrieval_alpha_06(query_index, celeba_image_embeddings, 10)

Query Index 0 (K=10):
  Average Recall@10: 0.2537
  Average Precision@10: 0.0389
Query Index 1 (K=10):
  Average Recall@10: 0.2322
  Average Precision@10: 0.0307
Query Index 2 (K=10):
  Average Recall@10: 0.1307
  Average Precision@10: 0.0160


KeyboardInterrupt: 

In [46]:
import torch
import torch.nn.functional as F

def not_even_slerp(query_index, query_source_image_index, celeba_image_embeddings, K, alpha):

  if type(query_source_image_index) is int:
    query_source_image_index = str(query_source_image_index)

  # Check that the source image index belongs to the example in that query
  # so that we will then be able to check its corresponding ground truth
  try:
    value = annotations[query_index]['ground_truth'][query_source_image_index]
  except (IndexError, KeyError, TypeError) as e:
    return {"error": str(e)}

  # get the original image embedding
  # This will have shape [512]
  query_source_image_embedding = celeba_image_embeddings[int(query_source_image_index)]

  combined_text_embedding = torch.zeros(512)
  for attribute in annotations[query_index]['query'].split(','):

    # extract the correct attribute
    attribute = attribute.strip()
    is_it_positive = attribute[0] =='+'
    attribute = attribute[1:]

    attribute_embedding = get_text_embedding(attribute, processor, model)
    attribute_embedding = F.normalize(attribute_embedding, dim=0)

    if is_it_positive:
        combined_text_embedding += attribute_embedding
    else:
        combined_text_embedding -= attribute_embedding

  combined_text_embedding = F.normalize(combined_text_embedding, dim=0)

  # normalize vectors (same as before)
  v = F.normalize(query_source_image_embedding, dim=0)
  w = combined_text_embedding

  # 🔥 REPLACED SLERP WITH RESIDUAL FUSION
  computed_embedding = F.normalize((1 - alpha) * v + alpha * w, dim=0)

  normalized_computed_embedding = computed_embedding.unsqueeze(0)

  normalized_celeba_embeddings = F.normalize(celeba_image_embeddings, p=2, dim=1)

  similarities = torch.mm(normalized_computed_embedding, normalized_celeba_embeddings.T)

  top_k_similarities, top_k_indices = torch.topk(similarities.squeeze(), K)

  return top_k_indices.tolist()

In [47]:
def metrics_per_query_not_even_slerp_retrieval_alpha_06(query_index, celeba_image_embeddings, K):
  total_recall = 0.0
  total_precision = 0.0
  evaluation_count = 0

  for test_image_index, _ in annotations[query_index]['ground_truth'].items():
      predictions = not_even_slerp(query_index, test_image_index, celeba_image_embeddings, K, 0.6)

      # Handle potential errors from simple_retrieval
      if isinstance(predictions, dict) and "error" in predictions:
            print(f"Warning: simple_retrieval returned an error for query_index {query_index}, source image '{test_image_index}': {predictions['error']}")
            continue # Skip this evaluation if an error occurred

      metrics = evaluate_retrieval(predictions, annotations[query_index]['ground_truth'][str(test_image_index)], K)

      total_recall += metrics[f"Recall@{K}"]
      total_precision += metrics[f"Precision@{K}"]
      evaluation_count += 1

  if evaluation_count > 0:
      average_recall = total_recall / evaluation_count
      average_precision = total_precision / evaluation_count
      print(f"Query Index {query_index} (K={K}):")
      print(f"  Average Recall@{K}: {average_recall:.4f}")
      print(f"  Average Precision@{K}: {average_precision:.4f}")
  else:
      print(f"Query Index {query_index} (K={K}): No evaluations performed.")

In [48]:
for query_index in range(len(annotations)):
  metrics_per_query_not_even_slerp_retrieval_alpha_06(query_index, celeba_image_embeddings, 10)

Query Index 0 (K=10):
  Average Recall@10: 0.2543
  Average Precision@10: 0.0389
Query Index 1 (K=10):
  Average Recall@10: 0.2454
  Average Precision@10: 0.0327
Query Index 2 (K=10):
  Average Recall@10: 0.1302
  Average Precision@10: 0.0161
Query Index 3 (K=10):
  Average Recall@10: 0.0251
  Average Precision@10: 0.0035
Query Index 4 (K=10):
  Average Recall@10: 0.0530
  Average Precision@10: 0.0070


KeyboardInterrupt: 